In [1]:
import os

In [2]:
%pwd

'c:\\Users\\sagal\\OneDrive\\Desktop\\Let us build\\emotion_detection\\research'

In [3]:
os.chdir('../')

In [4]:
%pwd

'c:\\Users\\sagal\\OneDrive\\Desktop\\Let us build\\emotion_detection'

In [7]:
#first step update the entity

from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen = True)
class ModelTrainerConfig:
    root_dir: Path
    train_data_path: Path
    test_data_path: Path
    model_path: Path
    val_split_size: float
    batch_size: int
    epochs: int
    patience: int
    lr_layer3: float
    lr_layer4: float
    lr_fc: float

In [8]:
from emotion_detection.constant import *
from emotion_detection.utils.common import read_yaml, create_directories

In [13]:
#config 2nd
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_model_trainer_config(self) -> ModelTrainerConfig:
        config = self.config.model_trainer
        params = self.params.ModelTrainerParams

        create_directories([config.root_dir])

        model_trainer_config = ModelTrainerConfig(
            root_dir=Path(config.root_dir),
            train_data_path=Path(config.train_data_path),
            test_data_path=Path(config.test_data_path),
            model_path=Path(config.model_path),
            val_split_size=float(params.val_split_size),
            batch_size=int(params.batch_size),
            epochs=int(params.epochs),
            patience=int(params.patience),
            lr_layer3=float(params.lr_layer3),
            lr_layer4=float(params.lr_layer4),
            lr_fc=float(params.lr_fc)
        )

        return model_trainer_config

In [10]:
import os
from emotion_detection.logging import logger

In [11]:
import os
import random
import torch
import torch.nn as nn
from torch.optim import Adam
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Subset, WeightedRandomSampler
from collections import Counter
from emotion_detection.logging import logger

In [14]:
#component 3rd

class ModelTrainer:
    def __init__(self, config: ModelTrainerConfig):
        self.config = config
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

    def train(self):
        logger.info(f"Using device: {self.device}")

        #transforms
        train_transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.Grayscale(num_output_channels=3),
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(10),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

        val_transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.Grayscale(num_output_channels=3),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

        #dataset from transform paths
        train_dir = str(self.config.train_data_path)
        
        train_dataset = datasets.ImageFolder(root=train_dir, transform=train_transform)
        val_dataset   = datasets.ImageFolder(root=train_dir, transform=val_transform)

        logger.info(f"Total base train images found: {len(train_dataset)}")

        #val split creation
        indices = list(range(len(train_dataset)))
        random.seed(42)
        random.shuffle(indices)

        split_size = int(self.config.val_split_size * len(indices))
        train_indices = indices[:split_size]
        val_indices   = indices[split_size:]

        train_subset = Subset(train_dataset, train_indices)
        val_subset   = Subset(val_dataset, val_indices)

        logger.info(f"Train size: {len(train_subset)} | Val size: {len(val_subset)}")

        #class balance and label counts
        train_labels = [train_dataset.targets[i] for i in train_indices]
        class_counts = Counter(train_labels)
        
        total = len(train_labels)
        class_weights = {cls: total / count for cls, count in class_counts.items()}
        sample_weights = [class_weights[label] for label in train_labels]

        sampler = WeightedRandomSampler(
            weights=sample_weights,
            num_samples=len(sample_weights),
            replacement=True
        )

        #dataloader configs
        train_loader = DataLoader(train_subset, batch_size=self.config.batch_size, sampler=sampler)
        val_loader   = DataLoader(val_subset,   batch_size=self.config.batch_size, shuffle=False)

        #network resnet18 TFL
        resnet = models.resnet18(weights="IMAGENET1K_V1")

        
        for name, param in resnet.named_parameters():
            if "layer3" in name or "layer4" in name or "fc" in name:
                param.requires_grad = True
            else:
                param.requires_grad = False

        #Apply Dropout(0.4) and target linear head for 7 emotion classes
        resnet.fc = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(resnet.fc.in_features, 7)
        )
        
        model = resnet.to(self.device)
        criterion = nn.CrossEntropyLoss()

        #Differential learning rates setup
        optimizer = Adam([
            {"params": model.layer3.parameters(), "lr": self.config.lr_layer3},
            {"params": model.layer4.parameters(), "lr": self.config.lr_layer4},
            {"params": model.fc.parameters(),     "lr": self.config.lr_fc}
        ])

        #Training & Validation Loop with Early Stopping
        best_val_acc     = 0.0
        patience_counter = 0

        logger.info("Starting model training pipeline stage...")

        for epoch in range(self.config.epochs):
            model.train()
            train_loss, train_correct = 0, 0

            for images, labels in train_loader:
                images, labels = images.to(self.device), labels.to(self.device)
                optimizer.zero_grad()
                outputs = model(images)
                loss    = criterion(outputs, labels)
                loss.backward()
                optimizer.step()

                train_loss    += loss.item()
                train_correct += (outputs.argmax(1) == labels).sum().item()

            train_acc  = train_correct / len(train_subset) * 100
            train_loss = train_loss / len(train_loader)

            model.eval()
            val_loss, val_correct = 0, 0

            with torch.no_grad():
                for images, labels in val_loader:
                    images, labels = images.to(self.device), labels.to(self.device)
                    outputs = model(images)
                    loss    = criterion(outputs, labels)

                    val_loss    += loss.item()
                    val_correct += (outputs.argmax(1) == labels).sum().item()

            val_acc  = val_correct / len(val_subset) * 100
            val_loss = val_loss / len(val_loader)

            logger.info(
                f"Epoch [{epoch+1}/{self.config.epochs}] "
                f"Train Loss: {train_loss:.4f} Train Acc: {train_acc:.2f}% | "
                f"Val Loss: {val_loss:.4f} Val Acc: {val_acc:.2f}%"
            )

            #check improvement and save
            if val_acc > best_val_acc:
                best_val_acc     = val_acc
                patience_counter = 0
                torch.save(model.state_dict(), str(self.config.model_path))
                logger.info(f" Best model saved (val acc: {val_acc:.2f}%)")
            else:
                patience_counter += 1
                logger.info(f"  No improvement ({patience_counter}/{self.config.patience})")

                if patience_counter >= self.config.patience:
                    logger.info(f"Early stopping triggered at epoch {epoch+1}")
                    break

        logger.info(f"Training Complete. Best validation accuracy achieved: {best_val_acc:.2f}%")

In [15]:
#last pipeline

try:
    config_manager = ConfigurationManager()
    model_trainer_config = config_manager.get_model_trainer_config()
    model_trainer = ModelTrainer(config=model_trainer_config)
    model_trainer.train()

except Exception as e:
    raise e

[2026-07-19 14:17:00,431: INFO: common: YAML file loaded successfully from: config\config.yaml]
[2026-07-19 14:17:00,442: INFO: common: YAML file loaded successfully from: params.yaml]
[2026-07-19 14:17:00,444: INFO: common: created directory at artifacts]
[2026-07-19 14:17:00,446: INFO: common: created directory at artifacts/model_trainer]
[2026-07-19 14:17:01,519: INFO: 2890399202: Using device: cuda]
[2026-07-19 14:17:01,700: INFO: 2890399202: Total base train images found: 12271]
[2026-07-19 14:17:01,710: INFO: 2890399202: Train size: 10430 | Val size: 1841]
[2026-07-19 14:17:02,417: INFO: 2890399202: Starting model training pipeline stage...]
[2026-07-19 14:19:00,220: INFO: 2890399202: Epoch [1/40] Train Loss: 1.8109 Train Acc: 28.70% | Val Loss: 1.6907 Val Acc: 32.81%]
[2026-07-19 14:19:00,385: INFO: 2890399202:  Best model saved (val acc: 32.81%)]
[2026-07-19 14:20:28,725: INFO: 2890399202: Epoch [2/40] Train Loss: 1.4856 Train Acc: 43.79% | Val Loss: 1.4207 Val Acc: 45.63%]
[20